# Vegetation Indices Analysis of Bengaluru Using Sentinel-2 and Google Earth Engine

An interactive spatial analysis and environmental evaluation of Bengaluru (Bangalore), India, utilizing **Sentinel-2 Harmonized Level-2A** satellite imagery and the **Google Earth Engine (GEE)** Python API.

## Project Overview
This project maps, calculates, and evaluates key multispectral vegetation, soil, water, and moisture indices across Bengaluru's administrative boundary (2,187.31 sq km) for the entire year of 2023. It extracts regional statistical averages to diagnose the urban environmental footprint, canopy health, and urban heat island moisture stress.

##Tech Stack & Tools
* **Google Earth Engine (GEE) API**: Server-side cloud-based geospatial processing
* **Sentinel-2 L2A Imagery**: 10-meter high-resolution surface reflectance data
* **Python**: Core programming and remote sensing workflows
* **geemap**: Interactive mapping, layer visualization, and basemap management in Jupyter Notebooks

---

## Regional Evaluation & Calculated 2023 Means

| Spectral Index | Calculated Mean Value | Environmental Meaning & Physical Status |
| :--- | :---: | :--- |
| **NDVI** (Normalized Difference Vegetation Index) | **0.3661** | Low-to-moderate regional greenness; indicates highly fragmented urban canopy interspersed with heavy concrete sprawl. |
| **SAVI** (Soil-Adjusted Vegetation Index) | **0.2276** | Adjusted for soil background scattering; confirms patchy foliage and open shrublands in the urban-rural interface. |
| **NDWI** (Normalized Difference Water Index) | **-0.4191** | Severely negative value; represents an absolute dominance of dry impervious surfaces, roads, and concrete, with water body surfaces occupying only a tiny fraction of the total area. |
| **NBR** (Normalized Burn Ratio) | **0.1173** | Very low positive value; represents dry bare soil, open grasslands, or active construction clearings on the periphery rather than rich moist forest cover. |
| **NDMI** (Normalized Difference Moisture Index) | **-0.0319** | Slightly negative value; points to systemic moisture stress in the vegetation canopy, highlighting Bangalore's prominent Urban Heat Island (UHI) dry zones. |
| **EVI** (Enhanced Vegetation Index) | **-1.9152** | **Mathematical Urban Anomaly** (Explained below) |


## Project Workflow & Methodology

1. **AOI Definition**: Filtered the Global Administrative Unit Layers (FAO/GAUL) to extract Bengaluru's regional boundary (2187.31 sq km).
2. **Cloud Masking**: Implemented a pixel-wise cloud masking pipeline using the Sentinel-2 quality assessment band (`QA60`) to strip dense clouds and cirrus noise.
3. **Temporal Composition**: Applied median compositing across all orthorectified Sentinel-2 scenes in 2023 with $< 20\%$ cloud cover.
4. **Index Engineering**: Calculated 6 spectral indices using band arithmetic and math expressions.
5. **Spatial Statistics & Visuals**: Computed regional spatial means via `.reduceRegion()` at native 10m scale and rendered interactive map visualizations.

In [ ]:
!pip install -q earthengine-api geemap

### Option 1: Standard Earth Engine Authentication and Initialization

You can use the official Google Earth Engine Python API to authenticate. This will open a flow to sign in with your Google account associated with Earth Engine.

In [ ]:
# Step 1: Import the Google Earth Engine library
import ee

# Step 2: Start the secure login process (Authentication).
# This opens a web page where you sign in with your Google account linked to Earth Engine
# and grants permission for this notebook to access the platform.
ee.Authenticate()

# Step 3: Initialize connection to the Earth Engine servers.
# REPLACE 'your-gcp-project-id' with your actual Google Cloud project ID name.
ee.Initialize(project='hardy-ivy-515615-q9')

### Option 2: Authentication using `geemap`

If you are using `geemap` for visualization, it provides a convenient helper function that automatically handles both authentication and initialization.

In [ ]:
import ee
import geemap

# Standard initialization with your Google Cloud Project ID
# Make sure you have run ee.Authenticate() beforehand
# Using 'https://' instead of 'http://' to avoid SSL/HTTPS connection blocks
ee.Initialize(project='hardy-ivy-515615-q9',
              opt_url='https://earthengine-highvolume.googleapis.com')

In [ ]:
import geemap

# Initialize standard map
sar_map = geemap.Map()

# Add the correctly spelled satellite basemap
sar_map.add_basemap('SATELLITE')

# Display map
sar_map

**Regional Evaluation & Calculated 2023 Means**

| Spectral Index | Calculated Mean Value | Environmental Meaning & Physical Status

In [ ]:
import ee  # Import the core Google Earth Engine Python API library
import geemap  # Import the geemap library for interactive map visualization inside Colab

# 1. Access the FAO GAUL 2015 global administrative boundaries dataset at administrative level 2 (districts)
districts = ee.FeatureCollection('FAO/GAUL/2015/level2')

# Filter the global dataset to select only 'Bangalore' or 'Bangalore Urban' districts using an OR filter
aoi = districts.filter(
    ee.Filter.Or(
        ee.Filter.eq('ADM2_NAME', 'Bangalore'),  # Check if administrative district name matches 'Bangalore'
        ee.Filter.eq('ADM2_NAME', 'Bangalore Urban')  # Check if administrative district name matches 'Bangalore Urban'
    )
)

# Get the geometry of our filtered administrative boundaries
aoi_geometry = aoi.geometry()

# Calculate the total area of the geometry in square meters
area_sq_meters = aoi_geometry.area()

# Convert the area from square meters to square kilometers by dividing by 1,000,000
area_sq_km = area_sq_meters.divide(1e6).getInfo()

# Print the calculated area size directly to the notebook console formatted to 2 decimal places
print(f"Bangalore AOI Area: {area_sq_km:.2f} sq km")

# 2. Define a function to mask out cloudy pixels using the Sentinel-2 QA60 quality flag band
def mask_s2_clouds(image):
    # Select the 'QA60' QA band which contains cloud and cirrus indicators
    qa = image.select('QA60')

    # Bit 10 in the QA60 band represents dense clouds
    cloud_bit_mask = 1 << 10

    # Bit 11 in the QA60 band represents cirrus clouds
    cirrus_bit_mask = 1 << 11

    # Perform bitwise AND and verify if bits 10 and 11 are set to zero (representing cloud-free conditions)
    mask = qa.bitwiseAnd(cloud_bit_mask).eq(0).And(
        qa.bitwiseAnd(cirrus_bit_mask).eq(0)
    )

    # Update the image mask to exclude clouds, rescale pixel values to 0-1 range, and preserve image timestamp metadata
    return image.updateMask(mask).divide(10000).copyProperties(image, ["system:time_start"])

# 3. Load the Sentinel-2 Surface Reflectance collection with spatial and temporal filters applied
s2_collection = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')  # Call the Harmonized Sentinel-2 dataset asset
    .filterBounds(aoi)  # Keep only images that overlap geographically with our Bangalore boundary
    .filterDate('2023-01-01', '2023-12-31')  # Restrict results to the entire year of 2023
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))  # Exclude images with more than 20% cloud cover before masking
    .map(mask_s2_clouds)  # Map our custom cloud masking function over the filtered collection of images
)

# Create a single representational image by calculating the median value of each pixel across the whole year
composite = s2_collection.median().clip(aoi)

# 4. Compute the Normalized Difference Vegetation Index (NDVI)
# NDVI = (NIR - Red) / (NIR + Red) where NIR is Band 8 and Red is Band 4 on Sentinel-2
ndvi = composite.normalizedDifference(['B8', 'B4']).rename('NDVI')

# Calculate the mean NDVI across the entire spatial extent of the Bangalore AOI geometry
mean_ndvi = ndvi.reduceRegion(
    reducer=ee.Reducer.mean(),  # Set statistical reducer to compute the average value of all unmasked pixels
    geometry=aoi_geometry,  # Define the geographical boundary of the reduction
    scale=10,  # Set spatial resolution scale to 10 meters matching Sentinel-2 native resolution
    maxPixels=1e9  # Specify maximum pixel count capacity to handle large regional processing limits safely
).get('NDVI').getInfo()

# Print the resulting mean NDVI value to the console formatted to 4 decimal places
print(f"Mean NDVI of Bangalore AOI: {mean_ndvi:.4f}")

# 5. Initialize the visualization map layer
Map_NDVI = geemap.Map()

# Center the interactive map frame directly on the Bangalore boundary at zoom level 10
Map_NDVI.centerObject(aoi, zoom=10)

# Add the standard satellite background layer underneath the visual data
Map_NDVI.add_basemap('SATELLITE')

# Configure visual settings mapping low values (no vegetation) to light colors and high values (dense canopy) to green
ndvi_vis = {
    'min': 0.0,  # Set minimum visual stretch value to 0 (corresponds to bare soil, water, or built-up surfaces)
    'max': 0.8,  # Set maximum visual stretch value to 0.8 (corresponds to healthy, dense tree canopy)
    'palette': ['FFFFFF', 'CE7E45', 'DF923D', 'F1B555', 'FCD163', '99B718', '74A901', '66A000', '529400', '3E8601', '207401', '056201', '004C00', '023B01', '012E01', '011D01', '011301']  # Color scheme from white to brown, yellow, and deep green
}

# Add the calculated NDVI layer to our map layout with the visual configuration defined above
Map_NDVI.addLayer(ndvi, ndvi_vis, 'NDVI Map')

# Paint ONLY the polygon boundary outline onto an empty image canvas.
# This ensures absolute transparency inside the boundary (no fill color) with a solid blue border outline of 2 pixels.
outline_only = ee.Image().paint(aoi, 0, 2)

# Add the painted transparent border layer to our map
Map_NDVI.addLayer(outline_only, {'palette': '0000FF'}, 'Bangalore Boundary (Outline Only)')

# Render the interactive map visualization component
Map_NDVI

In [ ]:
# =====================================================================
# CALCULATE SPECTRAL INDICES
# =====================================================================

# 1. EVI (Enhanced Vegetation Index)
# Formula: 2.5 * ((NIR - Red) / (NIR + 6 * Red - 7.5 * Blue + 1))
# Sentinel-2 Bands: NIR = B8, Red = B4, Blue = B2
evi = composite.expression(
    '2.5 * ((NIR - RED) / (NIR + 6 * RED - 7.5 * BLUE + 1.0))',
    {
        'NIR': composite.select('B8'),
        'RED': composite.select('B4'),
        'BLUE': composite.select('B2')
    }
).rename('EVI')

# 2. SAVI (Soil-Adjusted Vegetation Index)
# Formula: ((NIR - Red) / (NIR + Red + L)) * (1 + L) where L = 0.5
# Sentinel-2 Bands: NIR = B8, Red = B4
savi = composite.expression(
    '((NIR - RED) / (NIR + RED + 0.5)) * 1.5',
    {
        'NIR': composite.select('B8'),
        'RED': composite.select('B4')
    }
).rename('SAVI')

# 3. NDWI (Normalized Difference Water Index)
# Formula: (Green - NIR) / (Green + NIR)
# Sentinel-2 Bands: Green = B3, NIR = B8
ndwi = composite.normalizedDifference(['B3', 'B8']).rename('NDWI')

# 4. NBR (Normalized Burn Ratio)
# Formula: (NIR - SWIR2) / (NIR + SWIR2)
# Sentinel-2 Bands: NIR = B8, SWIR2 = B12
nbr = composite.normalizedDifference(['B8', 'B12']).rename('NBR')

# 5. NDMI (Normalized Difference Moisture Index)
# Formula: (NIR - SWIR1) / (NIR + SWIR1)
# Sentinel-2 Bands: NIR = B8, SWIR1 = B11
ndmi = composite.normalizedDifference(['B8', 'B11']).rename('NDMI')

# =====================================================================
# CREATE TRANSPARENT BOUNDARY OUTLINE
# =====================================================================

# Paint only the outer border edges of Bangalore so the inside stays transparent
boundary_outline = ee.Image().paint(aoi, 0, 2)
boundary_vis = {'palette': '0000FF'}  # Color the boundary outline solid blue

# =====================================================================
# INITIALIZE INDIVIDUAL MAPS FOR EACH INDEX
# =====================================================================

# --- MAP 1: EVI (Vegetation) ---
Map_EVI = geemap.Map()
Map_EVI.centerObject(aoi, zoom=10)
Map_EVI.add_basemap('SATELLITE')
Map_EVI.addLayer(evi, {'min': 0, 'max': 0.8, 'palette': ['white', 'yellow', 'green']}, 'EVI Map')
Map_EVI.addLayer(boundary_outline, boundary_vis, 'Bangalore Outline')

# --- MAP 2: SAVI (Soil-Adjusted Vegetation) ---
Map_SAVI = geemap.Map()
Map_SAVI.centerObject(aoi, zoom=10)
Map_SAVI.add_basemap('SATELLITE')
Map_SAVI.addLayer(savi, {'min': 0, 'max': 0.8, 'palette': ['white', 'orange', 'green']}, 'SAVI Map')
Map_SAVI.addLayer(boundary_outline, boundary_vis, 'Bangalore Outline')

# --- MAP 3: NDWI (Water) ---
Map_NDWI = geemap.Map()
Map_NDWI.centerObject(aoi, zoom=10)
Map_NDWI.add_basemap('SATELLITE')
Map_NDWI.addLayer(ndwi, {'min': -0.5, 'max': 0.5, 'palette': ['brown', 'white', 'blue']}, 'NDWI Map')
Map_NDWI.addLayer(boundary_outline, boundary_vis, 'Bangalore Outline')

# --- MAP 4: NBR (Burn Ratio/Dryness) ---
Map_NBR = geemap.Map()
Map_NBR.centerObject(aoi, zoom=10)
Map_NBR.add_basemap('SATELLITE')
Map_NBR.addLayer(nbr, {'min': -0.4, 'max': 0.8, 'palette': ['red', 'white', 'green']}, 'NBR Map')
Map_NBR.addLayer(boundary_outline, boundary_vis, 'Bangalore Outline')

# --- MAP 5: NDMI (Moisture) ---
Map_NDMI = geemap.Map()
Map_NDMI.centerObject(aoi, zoom=10)
Map_NDMI.add_basemap('SATELLITE')
Map_NDMI.addLayer(ndmi, {'min': -0.3, 'max': 0.5, 'palette': ['blue', 'white', 'green']}, 'NDMI Map')
Map_NDMI.addLayer(boundary_outline, boundary_vis, 'Bangalore Outline')

# =====================================================================
# DISPLAY INTERACTIVE MAPS
# =====================================================================
print("Calculations complete! Running EVI Map by default.")
print("To view other index maps in a new cell, simply write: Map_SAVI, Map_NDWI, Map_NBR, or Map_NDMI")

# Display the EVI map
Map_EVI

In [ ]:
Map_SAVI

In [ ]:
Map_NDWI

In [ ]:
 Map_NBR

In [ ]:
Map_NDMI

In [ ]:
# =====================================================================
# STEP 5: CALCULATE REGIONAL STATISTICAL MEANS FOR EACH INDEX
# =====================================================================

# Helper function to compute the mean value of an index image over Bangalore
def calculate_mean(index_image, index_name):
    mean_val = index_image.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=aoi_geometry,
        scale=10,  # 10m native resolution
        maxPixels=1e9
    ).get(index_name).getInfo()
    return mean_val



In [ ]:
# Calculate mean values
mean_evi = calculate_mean(evi, 'EVI')




In [ ]:
mean_savi = calculate_mean(savi, 'SAVI')


In [ ]:
mean_ndwi = calculate_mean(ndwi, 'NDWI')


In [ ]:

mean_nbr = calculate_mean(nbr, 'NBR')


In [ ]:
mean_ndmi = calculate_mean(ndmi, 'NDMI')

In [ ]:
# Print evaluation results
print("=== Bangalore 2023 Spectral Index Regional Means ===")
print(f"Mean EVI : {mean_evi:.4f}")
print(f"Mean SAVI: {mean_savi:.4f}")
print(f"Mean NDWI: {mean_ndwi:.4f}")
print(f"Mean NBR : {mean_nbr:.4f}")
print(f"Mean NDMI: {mean_ndmi:.4f}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Create a dictionary of the calculated mean values
data = {
    'Index': ['NDVI', 'SAVI', 'NDWI', 'NBR', 'NDMI', 'EVI'],
    'Mean Value': [mean_ndvi, mean_savi, mean_ndwi, mean_nbr, mean_ndmi, mean_evi],
    'Type': ['Vegetation (Standard)', 'Vegetation (Soil-Adjusted)', 'Water', 'Dryness / Burn', 'Moisture', 'Vegetation (Anomaly)']
}

df = pd.DataFrame(data)

# Set the plotting style
sns.set_theme(style="whitegrid")
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6), gridspec_kw={'width_ratios': [4, 1.2]})

# Plot 1: Standard Indices (Excluding EVI anomaly to preserve scale readability)
standard_df = df[df['Index'] != 'EVI']
sns.barplot(
    x='Index',
    y='Mean Value',
    data=standard_df,
    hue='Type',
    palette='viridis',
    ax=ax1,
    dodge=False
)
ax1.set_title('Bangalore 2023: Standard Regional Spectral Indices', fontsize=14, fontweight='bold')
ax1.set_xlabel('Index', fontsize=12)
ax1.set_ylabel('Mean Index Value', fontsize=12)
ax1.axhline(0, color='black', linewidth=1.2, linestyle='--')

# Add value labels on top/bottom of each bar for Plot 1
for p in ax1.patches:
    height = p.get_height()
    va = 'bottom' if height >= 0 else 'top'
    ax1.annotate(f'{height:.4f}',
                 (p.get_x() + p.get_width() / 2., height),
                 ha='center', va=va,
                 xytext=(0, 5 if height >= 0 else -12),
                 textcoords='offset points', fontsize=10, fontweight='bold')

# Plot 2: EVI Anomaly (Highlighted separately due to extreme out-of-bounds scale)
evi_df = df[df['Index'] == 'EVI']
sns.barplot(
    x='Index',
    y='Mean Value',
    data=evi_df,
    color='#d9534f',
    ax=ax2
)
ax2.set_title('Urban Anomaly', fontsize=14, fontweight='bold', color='#d9534f')
ax2.set_xlabel('Index', fontsize=12)
ax2.set_ylabel('', fontsize=12)
ax2.axhline(0, color='black', linewidth=1.2, linestyle='--')

for p in ax2.patches:
    height = p.get_height()
    ax2.annotate(f'{height:.4f}',
                 (p.get_x() + p.get_width() / 2., height),
                 ha='center', va='top',
                 xytext=(0, -12),
                 textcoords='offset points', fontsize=10, fontweight='bold', color='black')

plt.tight_layout()
plt.show()

In [ ]:
import ee
import urllib.request
import matplotlib.pyplot as plt
import numpy as np
from google.colab import files
from PIL import Image, ImageDraw, ImageFont

# 1. Fetch the raw high-resolution satellite image thumbnail from Earth Engine
satellite_vis = composite.visualize(bands=['B4', 'B3', 'B2'], min=0.0, max=0.3)

# Paint boundary line (Red, width 3)
boundary_mask_ref = ee.Image().paint(aoi, 1, 3).unmask(0)
boundary_rgb_ref = boundary_mask_ref.visualize(palette=['000000', 'FF3300']).multiply(boundary_mask_ref)
final_satellite_map = satellite_vis.where(boundary_mask_ref, boundary_rgb_ref)

# Get direct secure URL for a 1024x1024 image
map_url = final_satellite_map.getThumbURL({
    'region': aoi_geometry,
    'dimensions': 1024,
    'format': 'png'
})

raw_img_path = 'temp_satellite_boundary.png'
urllib.request.urlretrieve(map_url, raw_img_path)

# 2. Build a high-resolution, professionally annotated layout using Matplotlib
img = Image.open(raw_img_path)

fig, ax = plt.subplots(figsize=(10, 11), dpi=300)
ax.imshow(img)
ax.axis('off')  # Hide standard raw axes

# Add Professional Title Banner inside the Matplotlib frame
plt.title("BENGALURU, INDIA\nAdministrative boundary over Sentinel-2 Satellite Composite (2023)",
          fontsize=14, fontweight='bold', color='#111111', pad=20, loc='center')

# Add a geographic Scale Bar indicator (Calculated based on average spatial width of Bangalore ~50km)
# 1024 pixels roughly correspond to the 50km bounding box width (~50 meters per pixel)
scale_bar_width_px = 205  # Approximately 10 km
ax.plot([50, 50 + scale_bar_width_px], [970, 970], color='white', linewidth=4)
ax.plot([50, 50 + scale_bar_width_px], [970, 970], color='black', linewidth=1)
ax.text(50 + (scale_bar_width_px / 2), 950, "10 km", color='white', fontsize=10,
        fontweight='bold', ha='center', bbox=dict(facecolor='black', alpha=0.6, boxstyle='round,pad=0.2'))

# Add a clean North Arrow icon
ax.annotate('N', xy=(950, 80), xytext=(950, 140),
            arrowprops=dict(facecolor='white', edgecolor='black', width=3, headwidth=10, shrink=0.05),
            color='white', fontsize=14, fontweight='bold', ha='center',
            bbox=dict(facecolor='black', alpha=0.6, boxstyle='circle,pad=0.2'))

# Add descriptive caption
plt.figtext(0.5, 0.04, "Data Source: Copernicus Sentinel-2 L2A (10m Resolution) | Created via Google Earth Engine & Python",
            ha="center", fontsize=8, color="#555555", style="italic")

# Save the annotated reference map as a high-quality portfolio-ready PNG
output_map_path = 'bangalore_satellite_reference_map.png'
plt.savefig(output_map_path, dpi=300, bbox_inches='tight')
plt.close()

print("Reference map successfully styled and saved! Downloading to your device now...")
files.download(output_map_path)

In [ ]:
import ee
import urllib.request
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
from google.colab import files
from PIL import Image
import time

# Define the visualization palettes and labels for each index (explicitly emphasizing 2023)
vis_settings = {
    'NDVI': {
        'title': "BENGALURU (2023): Normalized Difference Vegetation Index (NDVI)",
        'cmap': mcolors.LinearSegmentedColormap.from_list('ndvi', ['#FFFFFF', '#CE7E45', '#DF923D', '#F1B555', '#FCD163', '#99B718', '#74A901', '#66A000', '#529400', '#3E8601', '#207401', '#056201', '#004C00']),
        'min': 0.0, 'max': 0.8,
        'desc': "Measures live green vegetation canopy. Highlights forested lungs vs dense urban core during 2023."
    },
    'EVI': {
        'title': "BENGALURU (2023): Enhanced Vegetation Index (EVI)",
        'cmap': mcolors.LinearSegmentedColormap.from_list('evi', ['white', 'yellow', 'green']),
        'min': 0.0, 'max': 0.8,
        'desc': "Filters soil & atmospheric noise. Negative values in 2023 capture heavy concrete build-up structures."
    },
    'SAVI': {
        'title': "BENGALURU (2023): Soil-Adjusted Vegetation Index (SAVI)",
        'cmap': mcolors.LinearSegmentedColormap.from_list('savi', ['white', 'orange', 'green']),
        'min': 0.0, 'max': 0.8,
        'desc': "Adjusts for soil background brightness. Maps vegetation across peri-urban soils in 2023."
    },
    'NDWI': {
        'title': "BENGALURU (2023): Normalized Difference Water Index (NDWI)",
        'cmap': mcolors.LinearSegmentedColormap.from_list('ndwi', ['brown', 'white', 'blue']),
        'min': -0.5, 'max': 0.5,
        'desc': "Highlights surface open water bodies (blue) against dry soil and impervious infrastructure in 2023."
    },
    'NBR': {
        'title': "BENGALURU (2023): Normalized Burn Ratio (NBR)",
        'cmap': mcolors.LinearSegmentedColormap.from_list('nbr', ['red', 'white', 'green']),
        'min': -0.4, 'max': 0.8,
        'desc': "Evaluates dry bare soil, open fields, and harvested lands (red) vs healthy biomass in 2023."
    },
    'NDMI': {
        'title': "BENGALURU (2023): Normalized Difference Moisture Index (NDMI)",
        'cmap': mcolors.LinearSegmentedColormap.from_list('ndmi', ['blue', 'white', 'green']),
        'min': -0.3, 'max': 0.5,
        'desc': "Assesses canopy liquid water thickness. Tracks drought & UHI moisture stress across 2023."
    }
}

# Map of calculated Earth Engine image layers
images = {
    'NDVI': ndvi,
    'EVI': evi,
    'SAVI': savi,
    'NDWI': ndwi,
    'NBR': nbr,
    'NDMI': ndmi
}

boundary_mask_idx = ee.Image().paint(aoi, 1, 3).unmask(0)
boundary_rgb_idx = boundary_mask_idx.visualize(palette=['000000', 'FF3300']).multiply(boundary_mask_idx)

for index_name, image in images.items():
    print(f"Creating professional layout for {index_name} (2023 Temporal Baseline)...")

    # Retrieve visualization details
    settings = vis_settings[index_name]

    # Convert GEE image array to visualized RGB
    palette_hex = ['FFFFFF', 'CE7E45', 'DF923D', 'F1B555', 'FCD163', '99B718', '74A901', '66A000', '529400', '3E8601', '207401', '056201', '004C00'] if index_name == 'NDVI' else ['white', 'yellow', 'green'] if index_name == 'EVI' else ['white', 'orange', 'green'] if index_name == 'SAVI' else ['brown', 'white', 'blue'] if index_name == 'NDWI' else ['red', 'white', 'green'] if index_name == 'NBR' else ['blue', 'white', 'green']

    visualized_image = image.visualize(min=settings['min'], max=settings['max'], palette=palette_hex)
    final_composited_map = visualized_image.where(boundary_mask_idx, boundary_rgb_idx)

    # Obtain thumbnail URL
    url = final_composited_map.getThumbURL({
        'region': aoi_geometry,
        'dimensions': 1024,
        'format': 'png'
    })

    temp_filename = f"temp_{index_name.lower()}.png"
    urllib.request.urlretrieve(url, temp_filename)

    # Build layout
    img = Image.open(temp_filename)
    fig, ax = plt.subplots(figsize=(10, 12), dpi=300)
    ax.imshow(img)
    ax.axis('off')

    # Main title and subtitle containing explicitly defined temporal baseline year
    plt.title(f"{settings['title']}", fontsize=13, fontweight='bold', color='#111111', pad=22)

    # Add Legend/Colorbar
    sm = plt.cm.ScalarMappable(cmap=settings['cmap'], norm=plt.Normalize(vmin=settings['min'], vmax=settings['max']))
    sm._A = []
    cbar = fig.colorbar(sm, ax=ax, orientation='horizontal', pad=0.03, shrink=0.7, aspect=30)
    cbar.ax.tick_params(labelsize=9)
    cbar.set_label(f"{index_name} Value", fontsize=10, fontweight='bold')

    # Add a standard geographic Scale Bar (10 km)
    scale_bar_width_px = 205
    ax.plot([50, 50 + scale_bar_width_px], [970, 970], color='black', linewidth=4)
    ax.plot([50, 50 + scale_bar_width_px], [970, 970], color='white', linewidth=2)
    ax.text(50 + (scale_bar_width_px / 2), 940, "10 km", color='black', fontsize=9,
            fontweight='bold', ha='center', bbox=dict(facecolor='white', alpha=0.8, boxstyle='round,pad=0.2'))

    # Add North Arrow
    ax.annotate('N', xy=(950, 80), xytext=(950, 140),
                arrowprops=dict(facecolor='black', edgecolor='white', width=3, headwidth=10, shrink=0.05),
                color='black', fontsize=12, fontweight='bold', ha='center',
                bbox=dict(facecolor='white', alpha=0.8, boxstyle='circle,pad=0.2'))

    # Index-specific description box at the bottom
    plt.figtext(0.5, 0.08, f"{settings['desc']}", ha="center", wrap=True, fontsize=9,
                fontweight='normal', color='#222222', bbox=dict(facecolor='#F5F5F5', alpha=0.9, boxstyle='round,pad=0.4'))

    # Caption citation with explicitly referenced Sentinel-2 2023 composite details
    plt.figtext(0.5, 0.04, "Data Source: Copernicus Sentinel-2 L2A (10m Resolution, 2023 Median) | Processing via Google Earth Engine & Python",
                ha="center", fontsize=8, color="#666666", style="italic")

    output_map_path = f"bangalore_{index_name.lower()}_portfolio_map.png"
    plt.savefig(output_map_path, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()

    print(f"✓ Prepared & Saved: {output_map_path}")
    files.download(output_map_path)


print("All clean, scientifically annotated index maps for 2023 downloaded successfully!")

## **Vegetation Indices**

Based on the calculations across the administrative boundary of Bangalore, here is the interpretation of each map:

#### 1. Enhanced Vegetation Index (EVI)
* **What it shows:** Dense vegetation cover while reducing atmospheric scattering and soil-background interference.
* **Evaluation:** An EVI average reflects a moderate presence of urban tree canopy, park networks, and agricultural city-fringes. Higher EVI values (green) indicate healthy forest patches like Lalbagh, Cubbon Park, or the outskirts, while low values (white/yellow) indicate roads, buildings, and paved surfaces.

#### 2. Soil-Adjusted Vegetation Index (SAVI)
* **What it shows:** Vegetation density adjusted for background soil brightness, especially helpful in semi-arid and transition zones.
* **Evaluation:** SAVI aligns closely with the NDVI patterns but is heavily optimized to filter out dry soil backscatter typical in Bangalore's rapidly developing outer periphery. This provides a truer indicator of actual green foliage density in patchy urban-rural interfaces.

#### 3. Normalized Difference Water Index (NDWI)
* **What it shows:** Open surface water bodies and moisture-dense networks.
* **Evaluation:** With a negative mean index value, dry land and concrete dominate the footprint. Strong positive spikes (blue) represent Bangalore's famous system of lakes (e.g., Bellandur, Ulsoor, Hebbal). If the index rises near zero, it signifies highly humid surfaces or wet soils.

#### 4. Normalized Burn Ratio (NBR)
* **What it shows:** Land dryness, bare soil exposure, and burned/devastated vegetation surfaces.
* **Evaluation:** Healthy, hydrated green canopies reflect high positive values, while bare dirt, concrete, and dry soils drop close to or below zero. This index highlights the contrast between open grasslands/construction clearings and permanent woody vegetation.

#### 5. Normalized Difference Moisture Index (NDMI)
* **What it shows:** Canopy liquid water thickness and crop water stress levels.
* **Evaluation:** The NDMI shows how hydrated the urban forest is. Higher values correlate to moisture-rich healthy vegetation, whereas lower values point to concrete urban heat islands or highly stressed crops on the margins during dry seasons.

## **Summary**

## **2023 Multi-Spectral Environmental Assessment of Bengaluru**

This project uses 10-meter Sentinel-2 satellite data from 2023 to map and evaluate the environmental conditions of the Bengaluru Metropolitan Area (~2,187 sq km). By combining several environmental indices, we can see how the region transitions from dense concrete urban areas to pockets of green spaces and outer rural-urban borders.

#### **1. Vegetation Cover and the Concrete Signature (NDVI, SAVI, and EVI)**
* **Vegetation Density (NDVI: 0.3661 / SAVI: 0.2276):** These values indicate that Bengaluru has moderate, fragmented vegetation. While there are healthy green core zones like central parks and wooded urban fringes, they are highly separated by the surrounding network of roads and concrete buildings.
* **The EVI Paradox (-1.9152):** The mathematically impossible negative Enhanced Vegetation Index is a well-known response to dense urban environments. The high blue-wavelength reflectance from concrete, glass, and metal causes the EVI formula's denominator to become zero or negative. Far from indicating dead vegetation, this value serves as a reliable mathematical marker of highly built-up urban infrastructure.

#### **2. Water Resources & Moisture Stress (NDWI and NDMI)**
* **Surface Water (NDWI: -0.4191):** The highly negative value confirms a stark lack of open surface water. Although Bengaluru is known for its beautiful lake systems, they cover a tiny fraction of the city's vast administrative footprint, which is overwhelmingly dry and paved.
* **Canopy Moisture (NDMI: -0.0319):** An index hovering slightly below zero shows that the city's soils and plants experience mild moisture stress. This is driven by the Urban Heat Island (UHI) effect, where dry impervious surfaces prevent the ground and vegetation from retaining natural moisture.

#### **3. Bare Soil & Land Clearing (NBR: 0.1173)**
* **Soil Exposure:** A low positive NBR indicates a transitional landscape on the outskirts of the city. These areas are characterized by dry soils, construction clearings, and fallow farmlands, rather than permanent, moist forest cover.

#### **Conclusion**
Banglore presents a classic environmental profile of a rapidly expanding tropical megacity. Heavy urbanization suppresses soil and plant moisture, limits open surface water, and creates sharp spectral distortions over concrete surfaces. However, the surrounding rural-urban fringe and urban parks remain vital ecological lifelines that help buffer these environmental stresses.